# YOLOv8n fine-tuning - Mendeley waste dataset

This Colab notebook is self-contained. It does not clone or import this repository. It prepares the Mendeley YOLO dataset and fine-tunes YOLOv8n for four project classes: `paper`, `plastic`, `glass`, and `residual`.

Mendeley dataset page: https://data.mendeley.com/datasets/2x69gjbcz6/2

Download the dataset from Mendeley, upload the downloaded zip here, then run the cells in order.

In [ ]:
!nvidia-smi
!pip install -q ultralytics pyyaml

## Runtime configuration

Adjust `SAMPLE_SIZE`, `EPOCHS`, and `BATCH` if Colab runs out of memory or you want a faster smoke test.

In [ ]:
from pathlib import Path

WORK_DIR = Path('/content/waste_yolov8n_colab')
RAW_DIR = WORK_DIR / 'mendeley_raw'
PREPARED_DIR = WORK_DIR / 'data/mendeley_yolo_4bins'
RUNS_DIR = WORK_DIR / 'runs/waste_yolov8'

SAMPLE_SIZE = 600
VAL_RATIO = 0.20
SEED = 42
EPOCHS = 30
BATCH = 8
IMG_SIZE = 640

WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Work directory:', WORK_DIR)

## Upload and extract Mendeley dataset

Use the Mendeley **Download All** button from the dataset page. Upload the zip file when this cell asks for it.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zip_path = Path(next(iter(uploaded)))
RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(RAW_DIR)

print('Extracted to:', RAW_DIR)
print('Example files:', list(RAW_DIR.rglob('*'))[:10])

## Prepare 4-class YOLO dataset

Mapping used by this notebook:

- `paper`: paper, cardboard
- `plastic`: plastic, metal
- `glass`: glass
- `residual`: organic waste, battery waste, e-waste, cloth, other waste

In [ ]:
import random
import shutil
from dataclasses import dataclass

CONTAINER_CLASSES = ('paper', 'plastic', 'glass', 'residual')
MENDELEY_CLASS_NAMES = (
    'plastic',
    'paper',
    'cardboard',
    'metal',
    'glass',
    'organic waste',
    'battery waste',
    'e-waste',
    'cloth',
    'other waste',
)
MENDELEY_TO_PROJECT = {
    'plastic': 'plastic',
    'paper': 'paper',
    'cardboard': 'paper',
    'metal': 'plastic',
    'glass': 'glass',
    'organic waste': 'residual',
    'battery waste': 'residual',
    'e-waste': 'residual',
    'cloth': 'residual',
    'other waste': 'residual',
}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


@dataclass(frozen=True)
class ClassMapping:
    source_id: int
    source_name: str
    project_id: int
    project_name: str


@dataclass(frozen=True)
class PreparationSummary:
    selected_images: int
    train_images: int
    val_images: int
    skipped_images: int


def convert_mendeley_class_id(source_id: int) -> ClassMapping:
    source_name = MENDELEY_CLASS_NAMES[source_id]
    project_name = MENDELEY_TO_PROJECT[source_name]
    return ClassMapping(
        source_id=source_id,
        source_name=source_name,
        project_id=CONTAINER_CLASSES.index(project_name),
        project_name=project_name,
    )


def find_images(source_dir: Path) -> list[Path]:
    return sorted(path for path in source_dir.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS)


def label_for_image(source_dir: Path, image_path: Path) -> Path | None:
    candidates = [
        image_path.with_suffix('.txt'),
        source_dir / 'labels' / image_path.with_suffix('.txt').name,
        source_dir / 'labels' / image_path.relative_to(source_dir).with_suffix('.txt'),
    ]
    parts = list(image_path.relative_to(source_dir).parts)
    if 'images' in parts:
        parts[parts.index('images')] = 'labels'
        candidates.append(source_dir / Path(*parts).with_suffix('.txt'))

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def rewrite_label_rows(label_path: Path) -> str:
    rows = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if not parts:
            continue
        mapping = convert_mendeley_class_id(int(parts[0]))
        rows.append(' '.join([str(mapping.project_id), *parts[1:]]))
    return '\n'.join(rows) + ('\n' if rows else '')


def write_data_yaml(output_path: Path, train_path: str, val_path: str) -> None:
    lines = [f'train: {train_path}', f'val: {val_path}', 'names:']
    lines.extend(f'  {index}: {name}' for index, name in enumerate(CONTAINER_CLASSES))
    output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


def reset_output_tree(output_dir: Path) -> None:
    if output_dir.exists():
        shutil.rmtree(output_dir)
    for split in ('train', 'val'):
        (output_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (output_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)


def copy_items(items: list[tuple[Path, Path]], output_dir: Path, split: str) -> None:
    for image_path, label_path in items:
        target_image = output_dir / 'images' / split / image_path.name
        target_label = output_dir / 'labels' / split / label_path.name
        shutil.copy2(image_path, target_image)
        target_label.write_text(rewrite_label_rows(label_path), encoding='utf-8')


def prepare_mendeley_subset(
    source_dir: Path,
    output_dir: Path,
    sample_size: int = 600,
    val_ratio: float = 0.2,
    seed: int = 42,
) -> PreparationSummary:
    image_files = find_images(source_dir)
    random.Random(seed).shuffle(image_files)

    selected = []
    skipped = 0
    for image_path in image_files:
        label_path = label_for_image(source_dir, image_path)
        if label_path is None:
            skipped += 1
            continue
        selected.append((image_path, label_path))
        if len(selected) == sample_size:
            break

    if not selected:
        raise ValueError(f'No labeled images found under {source_dir}')

    val_count = max(1, round(len(selected) * val_ratio)) if len(selected) > 1 else 0
    train_items = selected[:-val_count] if val_count else selected
    val_items = selected[-val_count:] if val_count else []

    reset_output_tree(output_dir)
    copy_items(train_items, output_dir, 'train')
    copy_items(val_items, output_dir, 'val')
    write_data_yaml(output_dir / 'data.yaml', 'images/train', 'images/val')

    return PreparationSummary(
        selected_images=len(selected),
        train_images=len(train_items),
        val_images=len(val_items),
        skipped_images=skipped,
    )

In [ ]:
summary = prepare_mendeley_subset(
    source_dir=RAW_DIR,
    output_dir=PREPARED_DIR,
    sample_size=SAMPLE_SIZE,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

print(summary)
print((PREPARED_DIR / 'data.yaml').read_text())
print('Prepared images:', len(list((PREPARED_DIR / 'images').rglob('*.*'))))
print('Prepared labels:', len(list((PREPARED_DIR / 'labels').rglob('*.txt'))))

## Train YOLOv8n

The trained weights will be written under `runs/waste_yolov8/yolov8n_mendeley_4bins/weights/best.pt` inside the Colab work directory.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(PREPARED_DIR / 'data.yaml'),
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    project=str(RUNS_DIR),
    name='yolov8n_mendeley_4bins',
)

## Download trained weights

In [ ]:
best_weights = RUNS_DIR / 'yolov8n_mendeley_4bins/weights/best.pt'
print('best.pt:', best_weights)
files.download(str(best_weights))